# Generative AI 017 — Building a RAG System

"Chat with a transcript", end to end, with **no API key** — and then the part
most tutorials skip: **evaluating** it.

| Part | What we check |
|---|---|
| A | index a real transcript: **12 chunks**, stored at exactly **1.25×** |
| B | the manual steps and the chain send an **identical** prompt |
| C | context precision and recall, exactly — recall complete at **k=2** |
| D | chunk size: a metric that misleads, and **96%** of the text sent |

The transcript is in `data/rag_lesson_transcript.txt`. Needs `langchain-core`,
`scikit-learn`.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

## Part A — Ingest and index

In [ ]:
import pathlib, re
from langchain_core.documents import Document
from langchain_core.embeddings import Embeddings
from langchain_core.vectorstores import InMemoryVectorStore
from sklearn.feature_extraction.text import TfidfVectorizer

class TfidfEmbeddings(Embeddings):
    """Offline stand-in for an embedding model. Compares WORDS, not meaning."""
    def __init__(self, corpus):
        self.v = TfidfVectorizer(stop_words="english").fit(corpus)
    def embed_documents(self, texts):
        return self.v.transform(texts).toarray().tolist()
    def embed_query(self, text):
        return self.v.transform([text]).toarray()[0].tolist()

def recursive_split(text, chunk_size, chunk_overlap):
    """Lesson 013's splitter, with overlap: reuse the tail of the last chunk."""
    def split(t, seps):
        if len(t) <= chunk_size or not seps:
            return [t]
        sep, rest = seps[0], seps[1:]
        out = []
        for p in (list(t) if sep == "" else t.split(sep)):
            out.extend([p] if len(p) <= chunk_size else split(p, rest))
        return out
    chunks, current = [], ""
    for p in split(text, ["\n\n", "\n", ". ", " ", ""]):
        candidate = f"{current} {p}".strip() if current else p
        if len(candidate) <= chunk_size:
            current = candidate
        else:
            if current:
                chunks.append(current)
                tail = current[-chunk_overlap:] if chunk_overlap else ""
                current = f"{tail} {p}".strip() if len(tail) + len(p) < chunk_size else p
            else:
                current = p
    if current:
        chunks.append(current)
    return chunks

# ---- ingestion: any long text works. Here, a lesson transcript on disk. ----
# The transcript of this course's lesson on RAG, shipped in data/.
for candidate in ("../data/rag_lesson_transcript.txt", "data/rag_lesson_transcript.txt",
                  "practice/data/rag_lesson_transcript.txt"):
    if pathlib.Path(candidate).exists():
        text = pathlib.Path(candidate).read_text(encoding="utf-8")
        break

# ---- split, embed, store ------------------------------------------------------
chunks = recursive_split(text, chunk_size=1000, chunk_overlap=200)
store = InMemoryVectorStore(embedding=TfidfEmbeddings(chunks))
store.add_documents([Document(page_content=c, metadata={"chunk": i})
                     for i, c in enumerate(chunks)])

print(len(text), "characters ->", len(chunks), "chunks")
print(f"stored {sum(len(c) for c in chunks) / len(text):.2f}x the transcript")
# 7720 characters -> 12 chunks
# stored 1.25x the transcript   <- exactly lesson 013's 1/(1 - 0.2)

In [ ]:
assert len(text) == 7720 and len(chunks) == 12
factor = sum(len(c) for c in chunks) / len(text)
assert abs(factor - 1 / (1 - 0.2)) < 0.02        # lesson 013's prediction
print(f"overlap cost {factor:.2f}x - predicted {1/(1-0.2):.2f}x")

## Part B — Manual steps against one chain

In [ ]:
from langchain_core.language_models.fake_chat_models import FakeListChatModel
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnablePassthrough

retriever = store.as_retriever(search_kwargs={"k": 4})
prompt = PromptTemplate.from_template(
    "You are a helpful assistant. Answer ONLY from the provided transcript "
    "context. If the context is insufficient, just say you don't know.\n\n"
    "{context}\nQuestion: {question}")

class EchoModel(FakeListChatModel):          # returns the prompt it was given
    def _call(self, messages, stop=None, run_manager=None, **kwargs):
        return messages[-1].content

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

question = "What is an emergent property?"

# ---- by hand: retrieve, format, fill, generate -----------------------------
docs = retriever.invoke(question)
manual = EchoModel(responses=["x"]).invoke(
    prompt.invoke({"context": format_docs(docs), "question": question})).content

# ---- one chain --------------------------------------------------------------
chain = (RunnableParallel({"context": retriever | RunnableLambda(format_docs),
                           "question": RunnablePassthrough()})
         | prompt | EchoModel(responses=["x"]) | StrOutputParser())

print(manual == chain.invoke(question))      # True - character for character

# Swap EchoModel for ChatOpenAI(model="gpt-4o-mini", temperature=0.2) and this
# is the lesson's system, answering questions about the transcript.

Same prompt, character for character. The chain is the same system with the
plumbing written once.

## Part C — Evaluate before you improve

Each question is paired with a phrase that answers it; a chunk is relevant if
it contains the phrase. Exact labels, no judge.

In [ ]:
# You must evaluate a RAG system to improve it. Each question is paired with a
# phrase from the transcript that answers it; a chunk is relevant if it
# contains that phrase - so the labels are exact, with no LLM judge.
QUESTIONS = [
    ("What is parametric knowledge?", "called parametric knowledge"),
    ("What does fine-tuning do to a model?", "re-trains it on a smaller, domain-specific dataset"),
    ("Which paper first described in-context learning?", "Language Models Are Few"),
    ("What is an emergent property?", "suddenly appears"),
    ("How many labelled pairs does supervised fine-tuning need?", "1k"),
    ("What instruction goes into a RAG prompt?", "only from the provided context"),
    ("Which vector stores run in the cloud?", "Pinecone, Weaviate, Milvus, Qdrant"),
    ("Why is RAG cheaper than fine-tuning?", "no training, no labeled dataset"),
]

def evaluate(store, chunks, k):
    precision = recall = 0.0
    for q, phrase in QUESTIONS:
        relevant = {i for i, c in enumerate(chunks) if phrase in c}
        got = [d.metadata["chunk"] for d in store.similarity_search(q, k=k)]
        hits = sum(g in relevant for g in got)
        precision += hits / k            # how much of the context is relevant
        recall += 1.0 if hits else 0.0   # was the answer retrieved at all
    return precision / len(QUESTIONS), recall / len(QUESTIONS)

for k in (1, 2, 4, 8):
    p, r = evaluate(store, chunks, k)
    print(f"k={k}  context precision {p:.2f}  context recall {r:.2f}")

# k=1  context precision 0.75  context recall 0.75
# k=2  context precision 0.56  context recall 1.00
# k=4  context precision 0.31  context recall 1.00   <- the lesson's setting
# k=8  context precision 0.17  context recall 1.00
#
# On this transcript recall is already complete at k=2. The lesson's k=4
# buys no recall and halves precision - the model reads twice as much
# irrelevant text. On a longer source the answer could differ; this table is
# how you find out instead of copying k=4 from a tutorial.

In [ ]:
p2, r2 = evaluate(store, chunks, 2)
p4, r4 = evaluate(store, chunks, 4)
assert r2 == 1.0 and r4 == 1.0          # every answer already retrieved at k=2
assert p4 < p2                           # k=4 only adds irrelevant text
print(f"k=2: precision {p2:.2f}, recall {r2:.2f}")
print(f"k=4: precision {p4:.2f}, recall {r4:.2f}  <- the lesson's setting")

> Retrieval uses TF-IDF, not a real embedder. Faithfulness and answer relevancy
> need a real model to generate answers, so they are **not** measured here.

## Part D — Chunk size, and a metric that misleads

In [ ]:
print(f"{'size':>6}{'chunks':>8}{'precision@4':>13}{'chars sent':>12}{'of text':>9}")
for size in (300, 1000, 2000):
    ch = recursive_split(text, size, size // 5)
    st = InMemoryVectorStore(embedding=TfidfEmbeddings(ch))
    st.add_documents([Document(page_content=c, metadata={"chunk": i})
                      for i, c in enumerate(ch)])
    p, r = evaluate(st, ch, 4)
    sent = sum(sum(len(d.page_content) for d in st.similarity_search(q, k=4))
               for q, _ in QUESTIONS) / len(QUESTIONS)
    print(f"{size:>6}{len(ch):>8}{p:>13.2f}{sent:>12,.0f}{sent/len(text):>8.0%}")

#   size  chunks  precision@4  chars sent  of text
#    300      40         0.25         980      13%
#   1000      12         0.31       3,087      40%
#   2000       5         0.34       7,402      96%
#
# Precision counted by CHUNKS rises with size - which looks like a reason to
# use big chunks and is not. At 2000 there are only 5 chunks, so k=4 returns
# 96% of the transcript: retrieval has stopped retrieving. A metric that
# counts chunks cannot compare settings that change what a chunk is.
# Measure what the model actually reads.

In [ ]:
ch = recursive_split(text, 2000, 400)
st = InMemoryVectorStore(embedding=TfidfEmbeddings(ch))
st.add_documents([Document(page_content=c, metadata={"chunk": i}) for i, c in enumerate(ch)])
sent = sum(sum(len(d.page_content) for d in st.similarity_search(q, k=4))
           for q, _ in QUESTIONS) / len(QUESTIONS)
assert sent / len(text) > 0.9
print(f"at chunk_size 2000, k=4 sends {sent/len(text):.0%} of the transcript")
print("- retrieval has stopped retrieving. Measure what the model reads.")

## What to take away

- A RAG system is lesson 016's four steps in one chain — and it sends the same
  prompt as doing them by hand.
- Overlap cost exactly **1.25×** on real text, as predicted.
- **Evaluate first.** Recall was complete at **k=2**; the lesson's k=4 halved
  precision for nothing.
- Precision counted by chunks misleads; at 2,000-character chunks k=4 sent
  **96%** of the transcript.

## Exercises

1. Write three harder questions whose answer phrase uses different words from
   the question. What happens to recall at k=2?
2. Find the chunk size that maximises *characters of answer per character
   sent*. Is it one of the four tried here?
3. Add chunk numbers to `format_docs` so an answer could cite them. How would
   you check a real model's citations were correct?
4. If you have an API key, swap in a real model and ask all eight questions.
   Do any answers go beyond the retrieved context?